# Cycle 3 : Test retrait corrélation + seuil custom


In [103]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import utils
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [104]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [105]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [106]:
preprocessor = utils.create_preprocessor()

In [109]:
from sklearn.model_selection import FixedThresholdClassifier
from technova_features import TechNovaFeatureEngineering
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline

dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)
random_forest_model = RandomForestClassifier(random_state=42, class_weight='balanced')
logistic_regression = LogisticRegression(random_state=42, max_iter=1000)
gradient_boosting_model = GradientBoostingClassifier(random_state=42)

models = [
    {'name' : 'LogisticRegression', 'model': logistic_regression},
    {'name' : 'RandomForestClassifier', 'model': random_forest_model},
    {'name' : 'GradientBoostingClassifier', 'model': gradient_boosting_model},
]

for model in models:
    print(model['name'])
    for seuil in [0.3, 0.5, 0.7]:
        print(f'SEUIL : {seuil}')
        threshold_model = FixedThresholdClassifier(
            estimator=model['model'],
            threshold=seuil,
            response_method="predict_proba"
        )

        pipeline = Pipeline([
            ('features', TechNovaFeatureEngineering()),
            ('preprocessor', preprocessor),
            ('model', threshold_model),
        ])
        utils.benchmark(pipeline, train_data)

LogisticRegression
SEUIL : 0.3
CrossValidation Results : 
{'fit_time': array([0.05963397, 0.04471016, 0.04808116, 0.04968715, 0.04758191]), 'score_time': array([0.00770211, 0.00739264, 0.00807881, 0.00741386, 0.00741506]), 'test_recall': array([0.65      , 0.66666667, 0.56410256, 0.6       , 0.775     ]), 'test_f1': array([0.59090909, 0.63414634, 0.54320988, 0.54545455, 0.65957447])}
Recall moyen : 0.6511538461538462
F1-Score moyen : 0.5946588644910734
Training Résults : 
Recall moyen : 0.5641025641025641
F1-Score moyen : 0.4444444444444444
[[217  38]
 [ 17  22]]
SEUIL : 0.5
CrossValidation Results : 
{'fit_time': array([0.0557282 , 0.04503298, 0.04781413, 0.0472672 , 0.04977322]), 'score_time': array([0.00761199, 0.00730085, 0.00799274, 0.00843692, 0.00720572]), 'test_recall': array([0.4       , 0.64102564, 0.35897436, 0.4       , 0.45      ]), 'test_f1': array([0.49230769, 0.72463768, 0.46666667, 0.50793651, 0.53731343])}
Recall moyen : 0.45000000000000007
F1-Score moyen : 0.54577239

In [108]:
pipeline, recall, f1 = utils.train(random_forest_model, preprocessor, train_data, 0.3)

TypeError: train() takes from 2 to 3 positional arguments but 4 were given

In [ ]:
print(pipeline['model'])

In [ ]:
# Affichage des features les plus importantes pour notre modele de prédiction de la consommation
importances = pipeline['model'].feature_importances_

noms_variables = preprocessor.get_feature_names_out()

df_importance = pd.DataFrame({
    'Variable': noms_variables,
    'Importance': importances
})

df_importance = df_importance.sort_values(by='Importance', ascending=False)


plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance',
    y='Variable',
    data=df_importance,
    hue='Variable',
    palette='viridis',
    legend=False
)
plt.title("Top 10 des variables les plus importantes")
plt.xlabel("Importance")
plt.ylabel("Variables")
plt.tight_layout()
plt.show()

revenu_mensuel
annee_par_experience
age
score_penibilite_transport
annee_experience_totale
annees_dans_l_entreprise
evolution_hierarchie_score
ratio_poste_actuel_anciennete
score_bien_etre
nombre_participation_pee
ratio_anciennete_total_experience
annees_dans_le_poste_actuel
annees_sous_responsable_actuel


In [ ]:
print(df_importance['Variable'].tolist())